
# YOLO26 vs Prompt-Aware YOLOE-26 on RDD2022-China  
## A controlled prompt-feasibility experiment for a future lightweight paper model

This notebook answers a narrow scientific question before modifying YOLO26 further:

> **Does an in-model vision-language prompt head improve road-damage detection, or do apparent gains come from extra training or prompt-insensitive behavior?**

It compares:

1. **Existing YOLO26s baseline from Google Drive**.
2. **YOLOE-26s detection model trained with rich road-damage prompts**.
3. The same trained prompt model evaluated using:
   - correct rich prompts,
   - plain class-name prompts,
   - deliberately shuffled prompts,
   - meaningless control prompts.
4. Optional matched-budget controls:
   - standard YOLO26s trained for the same number of epochs,
   - YOLOE-26s trained using plain class names.

The experiment reports:

- official Ultralytics mAP50 and mAP50–95;
- per-class AP;
- parameter count, GFLOPs, latency and peak T4 memory;
- paired ground-truth recall comparison;
- aspect-ratio recall for thin cracks;
- McNemar significance test;
- bootstrap confidence interval for recall improvement.

## Interpretation

This notebook uses an actual prompt-aware detection architecture rather than post-processing YOLO crops.  
However, **YOLOE-26 itself is not your novel paper contribution**. Treat this as a feasibility gate:

- If correct prompts outperform shuffled/nonsense prompts, prompt semantics are useful.
- If YOLOE improves but shuffled prompts perform similarly, the gain is probably architecture/pretraining rather than your descriptions.
- If prompts help, the next step is a smaller custom prompt-conditioned YOLO26 head with controlled ablations.


## 1. Install and verify the T4 runtime

In [ ]:

!pip -q install -U "ultralytics>=8.4.0" pandas scipy pyyaml pillow


In [ ]:

import gc
import json
import math
import random
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

from PIL import Image
from scipy.stats import binomtest
from ultralytics import YOLO, YOLOE
from ultralytics.models.yolo.yoloe import YOLOEPETrainer

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "Select Runtime > Change runtime type > T4 GPU before running this notebook."
)
print("GPU:", torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free/total: {free / 1e9:.2f}/{total / 1e9:.2f} GB")

DEVICE = 0
torch.backends.cudnn.benchmark = True


## 2. Mount Drive and configure the experiment

In [ ]:

from google.colab import drive
drive.mount("/content/drive")


In [ ]:

# ========================= USER CONFIG =========================
DRIVE_ZIP = Path("/content/drive/MyDrive/Research/Dataset/China_Subset.zip")
LOCAL_ROOT = Path("/content/China_Subset")

# Existing baseline produced by your original notebook.
BASELINE_CKPT = Path(
    "/content/drive/MyDrive/Research/runs_rdd2022/"
    "yolo26s_fresh/train/weights/best.pt"
)

# All new outputs remain separate from the baseline.
OUT_DIR = Path(
    "/content/drive/MyDrive/Research/runs_rdd2022/"
    "yoloe26_prompt_feasibility"
)

IMGSZ = 640
PROMPT_EPOCHS = 50
TRAIN_BATCH = 8       # change to 4 if your T4 runtime runs out of memory
VAL_BATCH = 8
WORKERS = 2
SEED = 0

# Evaluation thresholds used by the paired GT analysis.
PAIR_CONF = 0.25
PAIR_IOU = 0.50
BOOTSTRAP_SAMPLES = 5000

# Optional controls. They increase runtime substantially.
RUN_MATCHED_BUDGET_YOLO26 = False
RUN_PLAIN_PROMPT_TRAIN_CONTROL = False

# Set True to overwrite and retrain completed experiment folders.
FORCE_RETRAIN = False
# ===============================================================

PLAIN_NAMES = [
    "longitudinal crack",
    "transverse crack",
    "alligator crack",
    "pothole",
    "repair",
]

# Exactly one prompt per class, preserving class-ID order.
RICH_PROMPTS = [
    "a long narrow asphalt crack running parallel to the road and traffic direction",
    "a long narrow asphalt crack crossing the road perpendicular to the traffic direction",
    "a dense interconnected web of polygon-shaped alligator cracks on asphalt pavement",
    "an irregular depressed pothole cavity with broken or missing asphalt",
    "a repaired asphalt patch with different texture and visible patch boundaries",
]

# Wrong semantic descriptions assigned to the original class IDs.
SHUFFLE_ORDER = [3, 4, 1, 0, 2]
SHUFFLED_PROMPTS = [RICH_PROMPTS[i] for i in SHUFFLE_ORDER]

# Same number and order of classes, but intentionally non-informative text.
NONSENSE_PROMPTS = [
    "road damage category alpha",
    "road damage category beta",
    "road damage category gamma",
    "road damage category delta",
    "road damage category epsilon",
]

assert len(PLAIN_NAMES) == len(RICH_PROMPTS) == 5
assert len(set(SHUFFLE_ORDER)) == 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Baseline checkpoint:", BASELINE_CKPT)
print("Experiment folder:", OUT_DIR)


## 3. Extract the dataset and create prompt-specific dataset YAML files

In [ ]:

if not LOCAL_ROOT.exists():
    assert DRIVE_ZIP.exists(), f"Dataset zip not found: {DRIVE_ZIP}"
    local_zip = Path("/content/China_Subset.zip")
    shutil.copy2(DRIVE_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as archive:
        archive.extractall("/content")
    print("Dataset extracted to", LOCAL_ROOT)

assert (LOCAL_ROOT / "train_set" / "images").exists()
assert (LOCAL_ROOT / "valid_set" / "images").exists()
assert (LOCAL_ROOT / "test_set" / "images").exists()
assert BASELINE_CKPT.exists(), (
    f"Baseline checkpoint does not exist:\n{BASELINE_CKPT}\n"
    "Run the original baseline notebook first or correct the path."
)

def write_dataset_yaml(filename, names):
    path = LOCAL_ROOT / filename
    config = {
        "path": str(LOCAL_ROOT),
        "train": "train_set/images",
        "val": "valid_set/images",
        "test": "test_set/images",
        "nc": len(names),
        "names": list(names),
    }
    with open(path, "w") as file:
        yaml.safe_dump(config, file, sort_keys=False)
    return path

PLAIN_YAML = write_dataset_yaml("dataset_plain.yaml", PLAIN_NAMES)
RICH_YAML = write_dataset_yaml("dataset_rich_prompts.yaml", RICH_PROMPTS)
SHUFFLED_YAML = write_dataset_yaml("dataset_shuffled_prompts.yaml", SHUFFLED_PROMPTS)
NONSENSE_YAML = write_dataset_yaml("dataset_nonsense_prompts.yaml", NONSENSE_PROMPTS)

for split in ["train_set", "valid_set", "test_set"]:
    image_dir = LOCAL_ROOT / split / "images"
    label_dir = LOCAL_ROOT / split / "labels"
    n_images = len([
        p for p in image_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ])
    n_labels = len(list(label_dir.glob("*.txt")))
    print(f"{split:10s}: {n_images:5d} images | {n_labels:5d} labels")

print("\nPrompt YAML:", RICH_YAML)
print(yaml.safe_load(open(RICH_YAML)))



## 4. Validate the existing Drive baseline

This cell never retrains or changes the baseline checkpoint.


In [ ]:

def metrics_rows(metrics, method):
    rows = [{
        "method": method,
        "class_id": -1,
        "class_name": "ALL",
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    }]

    for class_id, name in metrics.names.items():
        rows.append({
            "method": method,
            "class_id": int(class_id),
            "class_name": PLAIN_NAMES[int(class_id)],
            "mAP50": float(metrics.box.ap50[int(class_id)]),
            "mAP50_95": float(metrics.box.maps[int(class_id)]),
            "precision": np.nan,
            "recall": np.nan,
        })
    return rows

gc.collect()
torch.cuda.empty_cache()

baseline_model = YOLO(str(BASELINE_CKPT))
baseline_metrics = baseline_model.val(
    data=str(PLAIN_YAML),
    split="test",
    imgsz=IMGSZ,
    batch=VAL_BATCH,
    device=DEVICE,
    verbose=False,
    plots=False,
)

baseline_rows = metrics_rows(baseline_metrics, "YOLO26s Drive baseline")
baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df.round(4))
baseline_df.to_csv(OUT_DIR / "baseline_official_metrics.csv", index=False)

del baseline_model, baseline_metrics
gc.collect()
torch.cuda.empty_cache()



## 5. Train the prompt-aware YOLOE-26s detection model

The official YOLOE detection workflow is used:

1. Build `yoloe-26s.yaml` as a detection model.
2. Load same-scale `yoloe-26s-seg.pt` pretrained weights.
3. Fine-tune with `YOLOEPETrainer`.
4. Use the rich class descriptions as the training vocabulary.

Training is skipped when `best.pt` already exists.


In [ ]:

PROMPT_RUN_DIR = OUT_DIR / "rich_prompt_train"
PROMPT_CKPT = PROMPT_RUN_DIR / "weights" / "best.pt"

if PROMPT_CKPT.exists() and not FORCE_RETRAIN:
    print("Reusing trained prompt model:", PROMPT_CKPT)
else:
    if FORCE_RETRAIN and PROMPT_RUN_DIR.exists():
        shutil.rmtree(PROMPT_RUN_DIR)

    gc.collect()
    torch.cuda.empty_cache()

    prompt_train_model = YOLOE("yoloe-26s.yaml")
    prompt_train_model.load("yoloe-26s-seg.pt")

    prompt_train_model.train(
        data=str(RICH_YAML),
        trainer=YOLOEPETrainer,
        project=str(OUT_DIR),
        name="rich_prompt_train",
        exist_ok=True,
        epochs=PROMPT_EPOCHS,
        imgsz=IMGSZ,
        batch=TRAIN_BATCH,
        workers=WORKERS,
        device=DEVICE,
        optimizer="SGD",
        lr0=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        mosaic=1.0,
        patience=0,
        cache=False,
        amp=True,
        plots=False,
        seed=SEED,
    )

    del prompt_train_model
    gc.collect()
    torch.cuda.empty_cache()

assert PROMPT_CKPT.exists(), f"Prompt training did not produce {PROMPT_CKPT}"
print("Prompt checkpoint:", PROMPT_CKPT)



## 6. Optional matched-budget standard YOLO26 control

Enable `RUN_MATCHED_BUDGET_YOLO26=True` to train an ordinary YOLO26s for the same number of epochs and settings as the prompt model.

This distinguishes prompt-model gains from differences in training budget or current package behavior.


In [ ]:

MATCHED_CKPT = OUT_DIR / "matched_budget_yolo26" / "weights" / "best.pt"

if RUN_MATCHED_BUDGET_YOLO26:
    if MATCHED_CKPT.exists() and not FORCE_RETRAIN:
        print("Reusing matched-budget YOLO26:", MATCHED_CKPT)
    else:
        if FORCE_RETRAIN and MATCHED_CKPT.parent.parent.exists():
            shutil.rmtree(MATCHED_CKPT.parent.parent)

        model = YOLO("yolo26s")
        model.train(
            data=str(PLAIN_YAML),
            project=str(OUT_DIR),
            name="matched_budget_yolo26",
            exist_ok=True,
            epochs=PROMPT_EPOCHS,
            imgsz=IMGSZ,
            batch=TRAIN_BATCH,
            workers=WORKERS,
            device=DEVICE,
            optimizer="SGD",
            lr0=0.01,
            momentum=0.937,
            weight_decay=0.0005,
            warmup_epochs=3,
            mosaic=1.0,
            patience=0,
            cache=False,
            amp=True,
            plots=False,
            seed=SEED,
        )
        del model
        gc.collect()
        torch.cuda.empty_cache()

    assert MATCHED_CKPT.exists()
    print("Matched-budget checkpoint:", MATCHED_CKPT)
else:
    print("Skipped. Set RUN_MATCHED_BUDGET_YOLO26=True for the stronger control.")



## 7. Optional YOLOE training control using only plain class names

Enable `RUN_PLAIN_PROMPT_TRAIN_CONTROL=True` to train an otherwise identical YOLOE-26s model with only the ordinary class names.

This is the cleanest training-time test of whether the richer descriptions help.


In [ ]:

PLAIN_PROMPT_CKPT = OUT_DIR / "plain_prompt_train" / "weights" / "best.pt"

if RUN_PLAIN_PROMPT_TRAIN_CONTROL:
    if PLAIN_PROMPT_CKPT.exists() and not FORCE_RETRAIN:
        print("Reusing plain-prompt YOLOE:", PLAIN_PROMPT_CKPT)
    else:
        if FORCE_RETRAIN and PLAIN_PROMPT_CKPT.parent.parent.exists():
            shutil.rmtree(PLAIN_PROMPT_CKPT.parent.parent)

        model = YOLOE("yoloe-26s.yaml")
        model.load("yoloe-26s-seg.pt")
        model.train(
            data=str(PLAIN_YAML),
            trainer=YOLOEPETrainer,
            project=str(OUT_DIR),
            name="plain_prompt_train",
            exist_ok=True,
            epochs=PROMPT_EPOCHS,
            imgsz=IMGSZ,
            batch=TRAIN_BATCH,
            workers=WORKERS,
            device=DEVICE,
            optimizer="SGD",
            lr0=0.01,
            momentum=0.937,
            weight_decay=0.0005,
            warmup_epochs=3,
            mosaic=1.0,
            patience=0,
            cache=False,
            amp=True,
            plots=False,
            seed=SEED,
        )
        del model
        gc.collect()
        torch.cuda.empty_cache()

    assert PLAIN_PROMPT_CKPT.exists()
    print("Plain-prompt checkpoint:", PLAIN_PROMPT_CKPT)
else:
    print("Skipped. Enable for the final paper ablation.")



## 8. Evaluate the trained prompt model under prompt interventions

All evaluations below use the **same trained checkpoint**. Only the configured text prompts change.

This is important:

- `correct rich` tests the intended method;
- `plain names` tests whether descriptions add value;
- `shuffled` deliberately assigns incorrect semantics to class IDs;
- `nonsense` checks whether arbitrary text works almost as well.

A real prompt effect should show a meaningful decline under shuffled or nonsense prompts.


In [ ]:

PROMPT_ABLATIONS = {
    "YOLOE-26s | correct rich prompts": (RICH_PROMPTS, RICH_YAML),
    "YOLOE-26s | plain class names": (PLAIN_NAMES, PLAIN_YAML),
    "YOLOE-26s | shuffled wrong prompts": (SHUFFLED_PROMPTS, SHUFFLED_YAML),
    "YOLOE-26s | nonsense prompts": (NONSENSE_PROMPTS, NONSENSE_YAML),
}

prompt_eval_rows = []

prompt_model = YOLOE(str(PROMPT_CKPT))

for method, (prompts, dataset_yaml) in PROMPT_ABLATIONS.items():
    print("\nEvaluating:", method)
    prompt_model.set_classes(prompts)

    metrics = prompt_model.val(
        data=str(dataset_yaml),
        split="test",
        imgsz=IMGSZ,
        batch=VAL_BATCH,
        device=DEVICE,
        verbose=False,
        plots=False,
    )
    prompt_eval_rows.extend(metrics_rows(metrics, method))
    del metrics
    gc.collect()
    torch.cuda.empty_cache()

prompt_eval_df = pd.DataFrame(prompt_eval_rows)
display(
    prompt_eval_df[prompt_eval_df.class_name == "ALL"]
    .sort_values("mAP50_95", ascending=False)
    .round(4)
)
prompt_eval_df.to_csv(OUT_DIR / "prompt_ablation_official_metrics.csv", index=False)

del prompt_model
gc.collect()
torch.cuda.empty_cache()


## 9. Evaluate optional control checkpoints

In [ ]:

control_rows = []

if RUN_MATCHED_BUDGET_YOLO26 and MATCHED_CKPT.exists():
    model = YOLO(str(MATCHED_CKPT))
    metrics = model.val(
        data=str(PLAIN_YAML),
        split="test",
        imgsz=IMGSZ,
        batch=VAL_BATCH,
        device=DEVICE,
        verbose=False,
        plots=False,
    )
    control_rows.extend(metrics_rows(metrics, "YOLO26s | matched training budget"))
    del model, metrics
    gc.collect()
    torch.cuda.empty_cache()

if RUN_PLAIN_PROMPT_TRAIN_CONTROL and PLAIN_PROMPT_CKPT.exists():
    model = YOLOE(str(PLAIN_PROMPT_CKPT))
    model.set_classes(PLAIN_NAMES)
    metrics = model.val(
        data=str(PLAIN_YAML),
        split="test",
        imgsz=IMGSZ,
        batch=VAL_BATCH,
        device=DEVICE,
        verbose=False,
        plots=False,
    )
    control_rows.extend(metrics_rows(metrics, "YOLOE-26s | trained with plain names"))
    del model, metrics
    gc.collect()
    torch.cuda.empty_cache()

control_df = pd.DataFrame(control_rows)
if len(control_df):
    display(control_df[control_df.class_name == "ALL"].round(4))
    control_df.to_csv(OUT_DIR / "optional_control_metrics.csv", index=False)
else:
    print("No optional controls were enabled.")



## 10. Unified baseline-versus-prompt comparison

The main comparison table contains:

- original Drive baseline;
- all prompt interventions;
- optional controls when enabled.


In [ ]:

all_metric_frames = [baseline_df, prompt_eval_df]
if len(control_df):
    all_metric_frames.append(control_df)

all_metrics_df = pd.concat(all_metric_frames, ignore_index=True)
overall_comparison = (
    all_metrics_df[all_metrics_df.class_name == "ALL"]
    .copy()
    .sort_values("mAP50_95", ascending=False)
)

base_map = float(
    overall_comparison.loc[
        overall_comparison.method == "YOLO26s Drive baseline",
        "mAP50_95",
    ].iloc[0]
)
base_map50 = float(
    overall_comparison.loc[
        overall_comparison.method == "YOLO26s Drive baseline",
        "mAP50",
    ].iloc[0]
)

overall_comparison["delta_mAP50_95_vs_drive_baseline"] = (
    overall_comparison.mAP50_95 - base_map
)
overall_comparison["delta_mAP50_vs_drive_baseline"] = (
    overall_comparison.mAP50 - base_map50
)

display(overall_comparison.round(4))
overall_comparison.to_csv(
    OUT_DIR / "baseline_vs_prompt_overall_comparison.csv",
    index=False,
)

per_class_comparison = all_metrics_df[all_metrics_df.class_id >= 0].copy()
per_class_comparison.to_csv(
    OUT_DIR / "baseline_vs_prompt_per_class_metrics.csv",
    index=False,
)



## 11. Prompt-effect diagnostic

These deltas distinguish a useful semantic prompt effect from a generic architecture effect.


In [ ]:

def overall_value(method, column):
    row = overall_comparison[overall_comparison.method == method]
    return float(row[column].iloc[0])

rich_method = "YOLOE-26s | correct rich prompts"
plain_method = "YOLOE-26s | plain class names"
shuffle_method = "YOLOE-26s | shuffled wrong prompts"
nonsense_method = "YOLOE-26s | nonsense prompts"

prompt_effect = pd.DataFrame([
    {
        "comparison": "Rich prompts minus Drive baseline",
        "delta_mAP50_95": overall_value(rich_method, "mAP50_95") - base_map,
        "delta_mAP50": overall_value(rich_method, "mAP50") - base_map50,
    },
    {
        "comparison": "Rich prompts minus plain-name inference",
        "delta_mAP50_95": (
            overall_value(rich_method, "mAP50_95")
            - overall_value(plain_method, "mAP50_95")
        ),
        "delta_mAP50": (
            overall_value(rich_method, "mAP50")
            - overall_value(plain_method, "mAP50")
        ),
    },
    {
        "comparison": "Rich prompts minus shuffled prompts",
        "delta_mAP50_95": (
            overall_value(rich_method, "mAP50_95")
            - overall_value(shuffle_method, "mAP50_95")
        ),
        "delta_mAP50": (
            overall_value(rich_method, "mAP50")
            - overall_value(shuffle_method, "mAP50")
        ),
    },
    {
        "comparison": "Rich prompts minus nonsense prompts",
        "delta_mAP50_95": (
            overall_value(rich_method, "mAP50_95")
            - overall_value(nonsense_method, "mAP50_95")
        ),
        "delta_mAP50": (
            overall_value(rich_method, "mAP50")
            - overall_value(nonsense_method, "mAP50")
        ),
    },
])

display(prompt_effect.round(4))
prompt_effect.to_csv(OUT_DIR / "prompt_effect_deltas.csv", index=False)



## 12. Model size, GFLOPs, T4 latency and peak memory

Latency is measured after warm-up using the same test images, image size, confidence threshold and batch size.

The prompt encoder is configured once before timing. The timing measures repeated detection inference, not the one-time prompt setup.


In [ ]:

TEST_IMAGES = sorted([
    p for p in (LOCAL_ROOT / "test_set" / "images").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
])
BENCH_IMAGES = TEST_IMAGES[: min(100, len(TEST_IMAGES))]

def safe_model_info(model):
    try:
        info = model.info(verbose=False, imgsz=IMGSZ)
        if info is not None and len(info) >= 4:
            layers, params, gradients, gflops = info[:4]
            return int(params), float(gflops)
    except Exception as exc:
        print("model.info warning:", type(exc).__name__, str(exc)[:120])

    params = sum(p.numel() for p in model.model.parameters())
    return int(params), np.nan

def benchmark_model(model, label, prompts=None):
    if prompts is not None:
        model.set_classes(prompts)

    # Warm-up.
    for image_path in BENCH_IMAGES[:5]:
        model.predict(
            str(image_path),
            imgsz=IMGSZ,
            conf=PAIR_CONF,
            device=DEVICE,
            verbose=False,
        )

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()
    for image_path in BENCH_IMAGES:
        model.predict(
            str(image_path),
            imgsz=IMGSZ,
            conf=PAIR_CONF,
            device=DEVICE,
            verbose=False,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    params, gflops = safe_model_info(model)
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    return {
        "method": label,
        "images": len(BENCH_IMAGES),
        "latency_ms_per_image": 1000 * elapsed / max(len(BENCH_IMAGES), 1),
        "fps": len(BENCH_IMAGES) / max(elapsed, 1e-9),
        "parameters": params,
        "GFLOPs": gflops,
        "peak_allocated_GB": peak_gb,
    }

efficiency_rows = []

model = YOLO(str(BASELINE_CKPT))
efficiency_rows.append(
    benchmark_model(model, "YOLO26s Drive baseline")
)
del model
gc.collect()
torch.cuda.empty_cache()

model = YOLOE(str(PROMPT_CKPT))
efficiency_rows.append(
    benchmark_model(
        model,
        "YOLOE-26s correct rich prompts",
        prompts=RICH_PROMPTS,
    )
)
del model
gc.collect()
torch.cuda.empty_cache()

if RUN_MATCHED_BUDGET_YOLO26 and MATCHED_CKPT.exists():
    model = YOLO(str(MATCHED_CKPT))
    efficiency_rows.append(
        benchmark_model(model, "YOLO26s matched training budget")
    )
    del model
    gc.collect()
    torch.cuda.empty_cache()

efficiency_df = pd.DataFrame(efficiency_rows)
display(efficiency_df.round(4))
efficiency_df.to_csv(OUT_DIR / "efficiency_comparison_t4.csv", index=False)



## 13. Load ground-truth boxes for paired detection analysis

Official mAP is the primary metric. The following paired analysis adds statistical evidence by asking, for each exact ground-truth object:

> Did the baseline detect it correctly, and did the prompt model detect it correctly?

A prediction counts as correct when it has the correct class and IoU ≥ `PAIR_IOU`.


In [ ]:

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def load_ground_truth(split="test_set"):
    image_dir = LOCAL_ROOT / split / "images"
    label_dir = LOCAL_ROOT / split / "labels"
    rows = []

    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() not in IMAGE_SUFFIXES:
            continue

        with Image.open(image_path) as image:
            width, height = image.size

        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.exists():
            continue

        text = label_path.read_text().strip()
        if not text:
            continue

        for object_index, line in enumerate(text.splitlines()):
            fields = line.split()
            if len(fields) < 5:
                continue

            class_id = int(float(fields[0]))
            xc, yc, bw, bh = map(float, fields[1:5])

            box_width = bw * width
            box_height = bh * height
            x1 = xc * width - box_width / 2
            y1 = yc * height - box_height / 2
            x2 = x1 + box_width
            y2 = y1 + box_height

            rows.append({
                "stem": image_path.stem,
                "image_path": str(image_path),
                "object_index": object_index,
                "class_id": class_id,
                "class_name": PLAIN_NAMES[class_id],
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "aspect_ratio": max(box_width, box_height)
                    / max(min(box_width, box_height), 1e-9),
            })

    return pd.DataFrame(rows)

gt_df = load_ground_truth()
gt_df["ar_bucket"] = pd.cut(
    gt_df.aspect_ratio,
    bins=[0, 2, 4, 8, np.inf],
    labels=["AR<2", "AR2-4", "AR4-8", "AR>=8"],
)
print("Ground-truth objects:", len(gt_df))
display(
    gt_df.groupby(["class_name", "ar_bucket"], observed=True)
    .size()
    .unstack(fill_value=0)
)


## 14. Generate and cache baseline and prompt-model predictions

In [ ]:

def predict_dataset(model, split, cache_path, prompts=None):
    cache_path = Path(cache_path)
    if cache_path.exists() and not FORCE_RETRAIN:
        print("Loading cached predictions:", cache_path)
        return pd.read_csv(cache_path)

    if prompts is not None:
        model.set_classes(prompts)

    image_paths = sorted([
        p for p in (LOCAL_ROOT / split / "images").iterdir()
        if p.suffix.lower() in IMAGE_SUFFIXES
    ])

    rows = []
    for start in range(0, len(image_paths), 32):
        batch_paths = image_paths[start:start + 32]
        results = model.predict(
            [str(path) for path in batch_paths],
            imgsz=IMGSZ,
            conf=PAIR_CONF,
            batch=4,
            device=DEVICE,
            verbose=False,
            stream=True,
        )

        for input_path, result in zip(batch_paths, results):
            boxes = result.boxes
            if boxes is None or len(boxes) == 0:
                continue

            xyxy = boxes.xyxy.detach().cpu().numpy()
            classes = boxes.cls.detach().cpu().numpy().astype(int)
            confidences = boxes.conf.detach().cpu().numpy()

            for prediction_index in range(len(xyxy)):
                rows.append({
                    "stem": input_path.stem,
                    "prediction_index": prediction_index,
                    "class_id": int(classes[prediction_index]),
                    "confidence": float(confidences[prediction_index]),
                    "x1": float(xyxy[prediction_index, 0]),
                    "y1": float(xyxy[prediction_index, 1]),
                    "x2": float(xyxy[prediction_index, 2]),
                    "y2": float(xyxy[prediction_index, 3]),
                })

        torch.cuda.empty_cache()

    dataframe = pd.DataFrame(rows)
    dataframe.to_csv(cache_path, index=False)
    print("Saved predictions:", cache_path, "| boxes:", len(dataframe))
    return dataframe

baseline_model = YOLO(str(BASELINE_CKPT))
baseline_pred_df = predict_dataset(
    baseline_model,
    "test_set",
    OUT_DIR / f"baseline_predictions_conf_{PAIR_CONF:.2f}.csv",
)
del baseline_model
gc.collect()
torch.cuda.empty_cache()

prompt_model = YOLOE(str(PROMPT_CKPT))
prompt_pred_df = predict_dataset(
    prompt_model,
    "test_set",
    OUT_DIR / f"prompt_predictions_conf_{PAIR_CONF:.2f}.csv",
    prompts=RICH_PROMPTS,
)
del prompt_model
gc.collect()
torch.cuda.empty_cache()


## 15. Greedy one-to-one matching and paired recall

In [ ]:

def box_iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)

    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])

    intersection = (
        np.clip(x2 - x1, 0, None)
        * np.clip(y2 - y1, 0, None)
    )
    area_a = (
        np.clip(a[:, 2] - a[:, 0], 0, None)
        * np.clip(a[:, 3] - a[:, 1], 0, None)
    )
    area_b = (
        np.clip(b[:, 2] - b[:, 0], 0, None)
        * np.clip(b[:, 3] - b[:, 1], 0, None)
    )
    union = area_a[:, None] + area_b[None, :] - intersection
    return intersection / np.clip(union, 1e-9, None)

def mark_matched_ground_truth(gt, predictions, column_name):
    result = gt.copy()
    result[column_name] = False

    for stem in result.stem.unique():
        for class_id in range(len(PLAIN_NAMES)):
            gt_part = result[
                (result.stem == stem)
                & (result.class_id == class_id)
            ]
            pred_part = predictions[
                (predictions.stem == stem)
                & (predictions.class_id == class_id)
            ].sort_values("confidence", ascending=False)

            if len(gt_part) == 0 or len(pred_part) == 0:
                continue

            gt_boxes = gt_part[["x1", "y1", "x2", "y2"]].to_numpy(float)
            pred_boxes = pred_part[["x1", "y1", "x2", "y2"]].to_numpy(float)
            ious = box_iou_matrix(gt_boxes, pred_boxes)

            candidates = []
            for gt_index in range(len(gt_boxes)):
                for pred_index in range(len(pred_boxes)):
                    if ious[gt_index, pred_index] >= PAIR_IOU:
                        candidates.append((
                            float(ious[gt_index, pred_index]),
                            gt_index,
                            pred_index,
                        ))

            candidates.sort(reverse=True)
            used_gt = set()
            used_pred = set()

            for _, local_gt_index, local_pred_index in candidates:
                if local_gt_index in used_gt or local_pred_index in used_pred:
                    continue
                used_gt.add(local_gt_index)
                used_pred.add(local_pred_index)
                original_gt_index = gt_part.index[local_gt_index]
                result.loc[original_gt_index, column_name] = True

    return result

paired_df = mark_matched_ground_truth(
    gt_df,
    baseline_pred_df,
    "baseline_detected",
)
paired_df = mark_matched_ground_truth(
    paired_df,
    prompt_pred_df,
    "prompt_detected",
)

paired_df.to_csv(OUT_DIR / "paired_gt_detection_results.csv", index=False)

paired_summary = pd.DataFrame([
    {
        "method": "YOLO26s Drive baseline",
        "n_gt": len(paired_df),
        "recall_at_fixed_threshold": paired_df.baseline_detected.mean(),
    },
    {
        "method": "YOLOE-26s correct rich prompts",
        "n_gt": len(paired_df),
        "recall_at_fixed_threshold": paired_df.prompt_detected.mean(),
    },
])

display(paired_summary.round(4))
paired_summary.to_csv(OUT_DIR / "paired_gt_recall_summary.csv", index=False)



## 16. McNemar test and bootstrap confidence interval

McNemar's test uses only objects where the two models disagree:

- baseline-only detections;
- prompt-only detections.

The bootstrap interval estimates uncertainty in the paired recall difference.


In [ ]:

baseline_only = int(
    (paired_df.baseline_detected & ~paired_df.prompt_detected).sum()
)
prompt_only = int(
    (~paired_df.baseline_detected & paired_df.prompt_detected).sum()
)
both = int(
    (paired_df.baseline_detected & paired_df.prompt_detected).sum()
)
neither = int(
    (~paired_df.baseline_detected & ~paired_df.prompt_detected).sum()
)

discordant = baseline_only + prompt_only
mcnemar_p = (
    binomtest(
        min(baseline_only, prompt_only),
        n=discordant,
        p=0.5,
        alternative="two-sided",
    ).pvalue
    if discordant > 0
    else 1.0
)

rng = np.random.default_rng(SEED)
base_values = paired_df.baseline_detected.to_numpy(np.float32)
prompt_values = paired_df.prompt_detected.to_numpy(np.float32)
n_objects = len(paired_df)

bootstrap_deltas = np.empty(BOOTSTRAP_SAMPLES, dtype=np.float32)
for sample_index in range(BOOTSTRAP_SAMPLES):
    indices = rng.integers(0, n_objects, size=n_objects)
    bootstrap_deltas[sample_index] = (
        prompt_values[indices].mean()
        - base_values[indices].mean()
    )

ci_low, ci_high = np.quantile(bootstrap_deltas, [0.025, 0.975])
observed_delta = prompt_values.mean() - base_values.mean()

significance_df = pd.DataFrame([{
    "both_detected": both,
    "baseline_only": baseline_only,
    "prompt_only": prompt_only,
    "neither_detected": neither,
    "baseline_recall": base_values.mean(),
    "prompt_recall": prompt_values.mean(),
    "recall_delta_prompt_minus_baseline": observed_delta,
    "bootstrap_95_CI_low": ci_low,
    "bootstrap_95_CI_high": ci_high,
    "mcnemar_exact_p_value": mcnemar_p,
}])

display(significance_df.round(5))
significance_df.to_csv(
    OUT_DIR / "paired_significance_results.csv",
    index=False,
)


## 17. Thin-crack and per-class paired recall

In [ ]:

def grouped_recall_table(group_column):
    rows = []
    for group_value, part in paired_df.groupby(group_column, observed=True):
        rows.append({
            group_column: str(group_value),
            "n_gt": len(part),
            "baseline_recall": part.baseline_detected.mean(),
            "prompt_recall": part.prompt_detected.mean(),
            "delta_prompt_minus_baseline": (
                part.prompt_detected.mean()
                - part.baseline_detected.mean()
            ),
            "baseline_only": int(
                (part.baseline_detected & ~part.prompt_detected).sum()
            ),
            "prompt_only": int(
                (~part.baseline_detected & part.prompt_detected).sum()
            ),
        })
    return pd.DataFrame(rows)

ar_comparison = grouped_recall_table("ar_bucket")
class_comparison = grouped_recall_table("class_name")

print("Aspect-ratio buckets:")
display(ar_comparison.round(4))

print("Per-class:")
display(class_comparison.round(4))

ar_comparison.to_csv(
    OUT_DIR / "paired_recall_by_aspect_ratio.csv",
    index=False,
)
class_comparison.to_csv(
    OUT_DIR / "paired_recall_by_class.csv",
    index=False,
)



## 18. Automatic interpretation

The rules below are deliberately conservative. They do not decide publication quality, but they prevent overclaiming.


In [ ]:

rich_map = overall_value(rich_method, "mAP50_95")
plain_map = overall_value(plain_method, "mAP50_95")
shuffle_map = overall_value(shuffle_method, "mAP50_95")
nonsense_map = overall_value(nonsense_method, "mAP50_95")

delta_vs_baseline = rich_map - base_map
delta_vs_plain = rich_map - plain_map
delta_vs_shuffle = rich_map - shuffle_map
delta_vs_nonsense = rich_map - nonsense_map

print("=== PROMPT FEASIBILITY VERDICT ===")

if delta_vs_baseline <= 0:
    print(
        "No overall prompt-model improvement over the existing baseline. "
        "Do not build the paper around prompts yet."
    )
else:
    print(
        f"The correct prompt model improves mAP50-95 by "
        f"{delta_vs_baseline:+.4f} over the Drive baseline."
    )

if delta_vs_shuffle > 0.005 and delta_vs_nonsense > 0.005:
    print(
        "Correct semantics outperform wrong/control text by more than 0.5 mAP points. "
        "This is evidence that prompt content matters."
    )
else:
    print(
        "Correct prompts do not clearly outperform shuffled/nonsense prompts. "
        "Any gain may come mainly from YOLOE pretraining or architecture, not the descriptions."
    )

if ci_low > 0 and mcnemar_p < 0.05:
    print(
        "The paired recall gain is statistically supported: "
        "bootstrap CI is above zero and McNemar p < 0.05."
    )
else:
    print(
        "The paired recall improvement is not yet statistically conclusive."
    )

if len(ar_comparison):
    thin_rows = ar_comparison[ar_comparison.ar_bucket.isin(["AR4-8", "AR>=8"])]
    if len(thin_rows) and thin_rows.delta_prompt_minus_baseline.mean() > 0:
        print("Prompt model improves average recall for high-aspect-ratio objects.")
    else:
        print(
            "Prompting does not solve the thin-object problem; "
            "a P2/high-resolution or directional module is still needed."
        )



## 19. Files to use in your research decision

The main outputs are written to:

```text
/content/drive/MyDrive/Research/runs_rdd2022/yoloe26_prompt_feasibility/
```

Important CSV files:

- `baseline_vs_prompt_overall_comparison.csv`
- `baseline_vs_prompt_per_class_metrics.csv`
- `prompt_effect_deltas.csv`
- `efficiency_comparison_t4.csv`
- `paired_significance_results.csv`
- `paired_recall_by_aspect_ratio.csv`
- `paired_recall_by_class.csv`

## Decision for the next model

### Build a lightweight semantic head when

- correct rich prompts improve over the Drive baseline;
- correct prompts clearly outperform shuffled and nonsense prompts;
- the paired confidence interval excludes zero;
- latency and parameter increases are acceptable.

### Prioritize geometry and high-resolution features when

- high-aspect-ratio recall remains weak;
- prompt ablations are insensitive to semantic content;
- improvements occur mainly for compact classes such as potholes and repairs.

### Strong final-paper ablations

For a publishable claim, enable both optional controls:

```python
RUN_MATCHED_BUDGET_YOLO26 = True
RUN_PLAIN_PROMPT_TRAIN_CONTROL = True
```

Run at least three random seeds and report mean ± standard deviation.
